# Key Findings Summary
### Eye-Tracking Study: BFS vs DFS Algorithm Visualization

117 participants (59 BFS, 58 DFS) across three experience groups watched an animated algorithm visualization while an eye-tracker recorded their gaze at 60 Hz.

**Two Areas of Interest (AOIs):**
- **Pseudocode panel** (left) — the step-by-step code
- **Geospatial map** (right) — the animated graph traversal

**Experience groups:**
- **G1** — No programming experience
- **G2** — Brief programming knowledge
- **G3** — Multiple years of coursework

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.family': 'sans-serif', 'font.size': 10})
sns.set_style('whitegrid')

DATA_DIR = Path('../')

ALG_COL = {'BFS': '#2166AC', 'DFS': '#D6604D'}
GRP_COL = {1: '#1B7837', 2: '#762A83', 3: '#E66101'}
GRP_LBL = {1: 'G1\n(no exp)', 2: 'G2\n(brief)', 3: 'G3\n(years)'}

METAL_FILES = [
    ('Group1_metalBFSData.xls', 'BFS', 'DE-bft.wmv', 1),
    ('Group1_metalDFSData.xls', 'DFS', 'DE-dft.wmv', 1),
    ('Group2_metalBFSData.xls', 'BFS', 'DE-bft.wmv', 2),
    ('Group2_metalDFSData.xls', 'DFS', 'DE-dft.wmv', 2),
    ('Group3_metalBFSData.xls', 'BFS', 'DE-bft.wmv', 3),
    ('Group3_metalDFSData.xls', 'DFS', 'DE-dft.wmv', 3),
]
STAT_KW = {'nan', 'mean', 'sum', 'std', 'median', '', 'all recordings'}

def get_col(df, metric, video, aoi):
    return next((c for c in df.columns
                 if metric in c and video in c and aoi in c
                 and c.endswith('_Mean') and 'Include Zeros' not in c), None)

records = []
for fname, algo, video, grp in METAL_FILES:
    raw = pd.read_excel(DATA_DIR / fname, engine='xlrd')
    raw = raw.rename(columns={raw.columns[0]: 'participant'})
    col = {
        'tfd_pseudo':  get_col(raw, 'Total Fixation Duration', video, 'Rectangle_'),
        'tfd_map':     get_col(raw, 'Total Fixation Duration', video, 'Rectangle 2_'),
        'fc_pseudo':   get_col(raw, 'Fixation Count',          video, 'Rectangle_'),
        'fc_map':      get_col(raw, 'Fixation Count',          video, 'Rectangle 2_'),
        'ttff_pseudo': get_col(raw, 'Time to First Fixation',  video, 'Rectangle_'),
        'ttff_map':    get_col(raw, 'Time to First Fixation',  video, 'Rectangle 2_'),
        'fix_before':  get_col(raw, 'Fixations Before',        video, 'Rectangle_'),
        'vc_pseudo':   get_col(raw, 'Visit Count',             video, 'Rectangle_'),
        'vc_map':      get_col(raw, 'Visit Count',             video, 'Rectangle 2_'),
        'ffd_pseudo':  get_col(raw, 'First Fixation Duration', video, 'Rectangle_'),
    }
    for _, row in raw.iterrows():
        p = str(row.iloc[0]).strip()
        if p.lower() in STAT_KW:
            continue
        records.append({
            'participant': p.split('-')[0].split('=')[0].strip(),
            'algorithm': algo, 'group': grp,
            **{k: pd.to_numeric(row.get(v), errors='coerce') if v else np.nan
               for k, v in col.items()}
        })

df = pd.DataFrame(records)
df['ratio']         = df['tfd_pseudo'] / (df['tfd_map']    + 1e-9)
df['scanner_index'] = df['vc_pseudo']  / (df['tfd_pseudo'] + 1e-9)
df['avg_fix_depth'] = df['tfd_pseudo'] / (df['fc_pseudo']  + 1e-9)
df['switching_rate']= (df['vc_pseudo'] + df['vc_map']) / (df['tfd_pseudo'] + df['tfd_map'] + 1e-9)
df['ratio_c']       = df['ratio'].where(df['ratio'] < 25)

print(f'Loaded {len(df)} records  (BFS={len(df[df.algorithm=="BFS"])}, DFS={len(df[df.algorithm=="DFS"])})')
df.groupby(['group', 'algorithm']).size().unstack()

---
## Finding 1 — Algorithm determines when you first look at the code

**Why it's interesting:**
- BFS viewers look at the pseudocode almost instantly (median 18.86 ms)
- DFS viewers wait much longer (median 25.47 ms) and make ~22 more fixations on the map first
- The effect size is enormous: r = 0.954 for TTFF, r = 0.782 for fixations-before

**What it might mean:**
- BFS expands suddenly across many nodes at once — viewers don't know what's happening and immediately consult the pseudocode to orient themselves
- DFS moves one node at a time — viewers can follow the map without needing to check the code first
- The animation's *structure* directly controls the first 2–3 seconds of attention — a design lever that has nothing to do with the viewer's expertise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Finding 1 — BFS Forces Earlier Pseudocode Contact', fontsize=13, fontweight='bold')

metrics = [
    ('ttff_pseudo', 'Time to First Fixation on Pseudocode (ms)'),
    ('fix_before',  'Fixations Before First Pseudocode Look'),
]

for ax, (metric, ylabel) in zip(axes, metrics):
    plot_data = []
    labels = []
    for grp in [1, 2, 3]:
        for algo in ['BFS', 'DFS']:
            vals = df[(df['group'] == grp) & (df['algorithm'] == algo)][metric].dropna()
            plot_data.append(vals.values)
            labels.append(f'G{grp}\n{algo}')

    positions = [1, 1.5, 2.5, 3, 4, 4.5]
    colors = []
    for grp in [1, 2, 3]:
        colors += [ALG_COL['BFS'], ALG_COL['DFS']]

    bp = ax.boxplot(plot_data, positions=positions, widths=0.35, patch_artist=True,
                    medianprops=dict(color='white', linewidth=2))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)

    ax.set_xticks([1.25, 2.75, 4.25])
    ax.set_xticklabels(['G1\n(no exp)', 'G2\n(brief)', 'G3\n(years)'])
    ax.set_ylabel(ylabel)

    bfs_patch = mpatches.Patch(color=ALG_COL['BFS'], label='BFS')
    dfs_patch = mpatches.Patch(color=ALG_COL['DFS'], label='DFS')
    ax.legend(handles=[bfs_patch, dfs_patch], fontsize=9)

    # Add overall medians as text
    bfs_all = df[df['algorithm'] == 'BFS'][metric].dropna()
    dfs_all = df[df['algorithm'] == 'DFS'][metric].dropna()
    u, p = stats.mannwhitneyu(bfs_all, dfs_all, alternative='two-sided')
    r_effect = 1 - (2 * u) / (len(bfs_all) * len(dfs_all))
    ax.set_title(f'BFS median={bfs_all.median():.1f}  |  DFS median={dfs_all.median():.1f}\n'
                 f'Mann-Whitney p<.001, r={abs(r_effect):.3f}', fontsize=9)

plt.tight_layout()
plt.show()

---
## Finding 2 — DFS produces deeper but delayed pseudocode reading

**Why it's interesting:**
- DFS viewers make fewer but *longer* individual fixations on pseudocode (avg 0.693s vs BFS 0.547s, p < .001, r = 0.450)
- They take longer to start, but when they do look at the code, they read it more carefully
- BFS viewers look at code earlier and more often, but in shorter bursts

**What it might mean:**
- DFS viewers are using the pseudocode as a reference, not a crutch — they need it less but use it more deliberately
- BFS viewers are *checking* the code (quick confirmations); DFS viewers are *reading* the code (slow comprehension)
- Two different cognitive strategies emerging from the same material — driven entirely by which algorithm they watched

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Finding 2 — DFS Produces Deeper Pseudocode Fixations', fontsize=13, fontweight='bold')

for ax, grp in zip(axes, [1, 2, 3]):
    for algo in ['BFS', 'DFS']:
        vals = df[(df['group'] == grp) & (df['algorithm'] == algo)]['avg_fix_depth'].dropna()
        vals = vals[vals < vals.quantile(0.95)]  # trim extreme outliers for display
        ax.hist(vals, bins=12, alpha=0.6, color=ALG_COL[algo], label=algo, density=True)

    bfs_v = df[(df['group'] == grp) & (df['algorithm'] == 'BFS')]['avg_fix_depth'].dropna()
    dfs_v = df[(df['group'] == grp) & (df['algorithm'] == 'DFS')]['avg_fix_depth'].dropna()
    ax.axvline(bfs_v.median(), color=ALG_COL['BFS'], linestyle='--', linewidth=2)
    ax.axvline(dfs_v.median(), color=ALG_COL['DFS'], linestyle='--', linewidth=2)

    ax.set_title(f'{GRP_LBL[grp]}\nBFS median={bfs_v.median():.3f}s  DFS={dfs_v.median():.3f}s', fontsize=9)
    ax.set_xlabel('Avg Fixation Depth on Pseudocode (s)')
    if grp == 1:
        ax.set_ylabel('Density')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Finding 3 — Expertise follows an inverted-U: intermediates read code most

**Why it's interesting:**
- Pseudocode/map ratio: G1 = 9.57, **G2 = 12.33**, G3 = 6.12
- You'd expect a monotonic increase (experts read code most) — but G3 experts have the *lowest* ratio
- The BFS/DFS gap in ratio is significant for novices (p = .040) but disappears entirely for G2 and G3

**What it might mean:**
- **G1 novices** don't have the schema to use pseudocode purposefully — they look at both panels but can't integrate them
- **G2 intermediates** lean heavily on pseudocode because they can read it but still need it as a crutch
- **G3 experts** have internalized the algorithm — they barely need the pseudocode and use the map more efficiently
- Experts aren't more engaged; they're more *efficient* — they extract the same information with less fixation time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Finding 3 — Expertise Inverted-U: Intermediates Read Code Most', fontsize=13, fontweight='bold')

# Left: ratio by group and algorithm
ax = axes[0]
for algo in ['BFS', 'DFS']:
    means, sems, xs = [], [], []
    for grp in [1, 2, 3]:
        vals = df[(df['group'] == grp) & (df['algorithm'] == algo)]['ratio_c'].dropna()
        means.append(vals.mean())
        sems.append(vals.sem())
        xs.append(grp)
    offset = -0.1 if algo == 'BFS' else 0.1
    ax.errorbar([x + offset for x in xs], means, yerr=sems,
                color=ALG_COL[algo], marker='o', linewidth=2, capsize=4, label=algo)

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['G1\n(no exp)', 'G2\n(brief)', 'G3\n(years)'])
ax.set_ylabel('Pseudocode / Map TFD Ratio (mean ± SE)')
ax.set_title('Ratio by group and algorithm')
ax.legend()

# Right: beeswarm-style strip by group
ax = axes[1]
for grp in [1, 2, 3]:
    vals = df[df['group'] == grp]['ratio_c'].dropna()
    jitter = np.random.uniform(-0.15, 0.15, size=len(vals))
    ax.scatter(np.full(len(vals), grp) + jitter, vals,
               color=GRP_COL[grp], alpha=0.5, s=30, edgecolors='none')
    ax.scatter(grp, vals.median(), color=GRP_COL[grp], s=150,
               marker='D', zorder=5, edgecolors='black', linewidth=1)

ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['G1\n(no exp)', 'G2\n(brief)', 'G3\n(years)'])
ax.set_ylabel('Pseudocode / Map TFD Ratio')
ax.set_title('Individual distributions (diamond = median)\nshows within-group variance')

plt.tight_layout()
plt.show()

---
## Finding 4 — Gaze *style* is a stable personal trait; attention *allocation* is algorithm-driven

**Why it's interesting:**
- 52 participants appeared in both BFS and DFS conditions — we can test what's consistent within a person
- **Stable across algorithms** (high within-person correlation):
  - Scanner index: r = 0.589, p < .001
  - Avg fixation depth: r = 0.627, p < .001
  - Switching rate: r = 0.569, p < .001
- **Not stable** (algorithm drives it):
  - Pseudocode/map ratio: r = 0.155, *ns*

**What it might mean:**
- *How* you look at things (fast/slow, deep/shallow) is a personal cognitive trait — like reading speed
- *Where* you direct attention depends on what you're watching, not who you are
- Implication: interventions that try to change where students look need to work at the design level (change the animation). Interventions targeting how they look need to work at the curriculum level (change the learner).

In [ ]:
# Build paired dataset (participants in both BFS and DFS)
bfs_df = df[df['algorithm'] == 'BFS'][['participant', 'group', 'ratio_c', 'scanner_index', 'avg_fix_depth', 'switching_rate']].copy()
dfs_df = df[df['algorithm'] == 'DFS'][['participant', 'group', 'ratio_c', 'scanner_index', 'avg_fix_depth', 'switching_rate']].copy()
bfs_df.columns = ['participant', 'group', 'bfs_ratio', 'bfs_scanner', 'bfs_depth', 'bfs_switching']
dfs_df.columns = ['participant', 'group', 'dfs_ratio', 'dfs_scanner', 'dfs_depth', 'dfs_switching']
paired = pd.merge(bfs_df, dfs_df, on=['participant', 'group']).replace([np.inf, -np.inf], np.nan)

metrics_to_compare = [
    ('bfs_ratio',    'dfs_ratio',    'Pseudocode/Map Ratio\n(NOT stable — algorithm-driven)'),
    ('bfs_scanner',  'dfs_scanner',  'Scanner Index\n(STABLE — personal trait)'),
    ('bfs_depth',    'dfs_depth',    'Avg Fixation Depth\n(STABLE — personal trait)'),
    ('bfs_switching','dfs_switching','Switching Rate\n(STABLE — personal trait)'),
]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('Finding 4 — Gaze Style Is a Trait; Where You Look Depends on the Algorithm', fontsize=12, fontweight='bold')

for ax, (bcol, dcol, title) in zip(axes, metrics_to_compare):
    sub = paired[[bcol, dcol, 'group']].dropna()
    if bcol == 'bfs_ratio':
        sub = sub[(sub[bcol] < 25) & (sub[dcol] < 25)]

    for grp in [1, 2, 3]:
        g = sub[sub['group'] == grp]
        ax.scatter(g[bcol], g[dcol], color=GRP_COL[grp], s=50, alpha=0.7,
                   edgecolors='white', linewidth=0.5, label=f'G{grp}')

    if len(sub) >= 3:
        m, b_ = np.polyfit(sub[bcol], sub[dcol], 1)
        xline = np.linspace(sub[bcol].min(), sub[bcol].max(), 100)
        ax.plot(xline, m * xline + b_, color='tomato', linewidth=1.8, linestyle='--')
        r, p = stats.spearmanr(sub[bcol], sub[dcol])
        sig = 'p<.001' if p < .001 else f'p={p:.3f}'
        ax.text(0.05, 0.95, f'r = {r:.3f}\n{sig}', transform=ax.transAxes,
                fontsize=9, va='top', color='black',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    # Diagonal reference line
    lims = [min(sub[bcol].min(), sub[dcol].min()), max(sub[bcol].max(), sub[dcol].max())]
    ax.plot(lims, lims, color='#ccc', linewidth=1, linestyle=':')

    ax.set_xlabel('BFS value', fontsize=9)
    ax.set_ylabel('DFS value', fontsize=9)
    ax.set_title(title, fontsize=9)
    if bcol == 'bfs_ratio':
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Finding 5 — High switching = fragmentation, not integration

**Why it's interesting:**
- Intuitively you might think someone who frequently switches between the code and the map is actively *integrating* both representations
- But high switching is strongly *negatively* correlated with code engagement (ratio: BFS r = −0.48, DFS r = −0.45) and with fixation depth (BFS r = −0.51, DFS r = −0.53)
- Same pattern in both algorithms

**What it might mean:**
- Switching is a sign of cognitive fragmentation, not synthesis — the viewer is lost and bouncing between panels without processing either
- Deep, purposeful engagement means fewer, longer fixations — not rapid scanning
- Animations that cause high switching rates may be creating confusion rather than facilitating integration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Finding 5 — High Switching Rate = Fragmentation, Not Integration', fontsize=13, fontweight='bold')

pairs = [
    (axes[0][0], 'BFS', 'switching_rate', 'ratio_c',       'Pseudocode/Map Ratio (clipped)'),
    (axes[0][1], 'DFS', 'switching_rate', 'ratio_c',       'Pseudocode/Map Ratio (clipped)'),
    (axes[1][0], 'BFS', 'switching_rate', 'avg_fix_depth', 'Avg Fixation Depth (s)'),
    (axes[1][1], 'DFS', 'switching_rate', 'avg_fix_depth', 'Avg Fixation Depth (s)'),
]

for ax, algo, xcol, ycol, ylabel in pairs:
    sub = df[df['algorithm'] == algo][[xcol, ycol]].replace([np.inf, -np.inf], np.nan).dropna()
    xv, yv = sub[xcol], sub[ycol]

    ax.scatter(xv, yv, color=ALG_COL[algo], s=60, alpha=0.75, edgecolors='white', linewidth=0.4)
    m, b_ = np.polyfit(xv, yv, 1)
    xline = np.linspace(xv.min(), xv.max(), 100)
    ax.plot(xline, m * xline + b_, color='#333', linewidth=1.8, linestyle='--')

    r, p = stats.spearmanr(xv, yv)
    sig = 'p<.001' if p < .001 else f'p={p:.3f}'
    ax.set_title(f'{algo} — Switching Rate vs {ylabel}\nSpearman r={r:.2f}, {sig}, n={len(sub)}', fontsize=9)
    ax.set_xlabel('Switching Rate (AOI visits/s)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)

plt.tight_layout()
plt.show()

---
## Finding 6 — Intermediates show parallel engagement; novices and experts don't
### *(Strongest finding — best candidate for a research paper)*

**Why it's interesting:**
- When you plot fixation count on pseudocode vs fixation count on the map, the *relationship* changes completely across experience groups:
  - **G1 novices**: near-zero correlation (r = −0.018) — the two AOIs are independent
  - **G2 intermediates**: positive correlation (r = +0.317, p = .049) — more code fixations *and* more map fixations go together
  - **G3 experts**: negative correlation (r = −0.276) — it's a trade-off, the two panels compete for attention
- The direction *reverses* between G2 and G3, not just shifts. This is not a monotonic trend — G2 is doing something qualitatively different.

**What it might mean:**
- **G1** can't use either representation purposefully — their fixations are driven by visual salience, not comprehension strategy
- **G2** has just enough schema to use both representations actively at once — the code helps them understand the map and vice versa. This is true *dual-coding*.
- **G3** has internalized the pseudocode — they only check it when they don't need the map, and vice versa. It becomes a strategic trade-off, not a parallel resource.
- **Implication**: integrative visualizations (ones that link code and map animations tightly) will benefit G2 most. G1 needs scaffolding first. G3 doesn't need the pseudocode panel at all.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle(
    'Finding 6 — Intermediates Show Parallel Engagement; Novices and Experts Don\'t\n'
    'Positive slope = both AOIs increase together. Negative slope = trade-off.',
    fontsize=12, fontweight='bold'
)

for ax, grp in zip(axes, [1, 2, 3]):
    sub = df[df['group'] == grp][['fc_pseudo', 'fc_map', 'algorithm']].dropna()

    for algo in ['BFS', 'DFS']:
        a = sub[sub['algorithm'] == algo]
        mk = 'o' if algo == 'BFS' else 's'
        ax.scatter(a['fc_pseudo'], a['fc_map'], color=ALG_COL[algo], marker=mk,
                   s=55, alpha=0.75, label=algo, edgecolors='white', linewidth=0.4)

    # Overall regression for this group
    xv, yv = sub['fc_pseudo'].values, sub['fc_map'].values
    m, b_ = np.polyfit(xv, yv, 1)
    xline = np.linspace(xv.min(), xv.max(), 100)
    ax.plot(xline, m * xline + b_, color=GRP_COL[grp], linewidth=2.5, zorder=5)

    r, p = stats.spearmanr(xv, yv)
    sig = '***' if p < .001 else '**' if p < .01 else '*' if p < .05 else '~' if p < .10 else 'ns'
    direction = 'PARALLEL ↗' if r > 0.1 else 'TRADE-OFF ↘' if r < -0.1 else 'INDEPENDENT'

    ax.set_title(f'{GRP_LBL[grp]}\nr = {r:.3f} {sig} — {direction}',
                 fontsize=11, color=GRP_COL[grp], fontweight='bold')
    ax.set_xlabel('Fixation Count — Pseudocode', fontsize=10)
    if grp == 1:
        ax.set_ylabel('Fixation Count — Map', fontsize=10)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Finding 7 — Expertise tightens the entire attentional system

**Why it's interesting:**
- For every pair of eye-tracking metrics, compute the Spearman correlation. Average the absolute values across all pairs.
- Result: G1 = 0.294, G2 = 0.330, G3 = 0.369 — a monotonic increase
- In novices, metrics are mostly independent. In experts, knowing one metric lets you predict most of the others.
- Specific couplings that *emerge* with expertise:
  - ratio ↔ scanner_index: G1 r = −0.04 → G3 r = −0.67
  - scanner_index ↔ tfd_map: G1 r = +0.07 → G3 r = +0.66
  - ratio ↔ switching_rate: G1 r = −0.22 → G3 r = −0.65

**What it might mean:**
- Novices don't have a coherent attention strategy — each metric is driven by different, unrelated forces
- Experts have a tightly coordinated system: their scanning rate, fixation depth, switching behavior, and allocation all move together as part of one unified strategy
- This is the eye-tracking signature of expertise: not just looking at the right things, but having an *integrated* attentional architecture

In [ ]:
NET_METRICS = ['ratio_c', 'scanner_index', 'avg_fix_depth', 'switching_rate',
               'tfd_pseudo', 'tfd_map', 'fc_pseudo', 'fc_map', 'fix_before', 'ttff_pseudo']
NET_LBL = ['ratio', 'scanner', 'depth', 'switching',
           'TFD\npseudo', 'TFD\nmap', 'FC\npseudo', 'FC\nmap', 'fix\nbefore', 'TTFF']

# Compute coupling strength per group
coupling = {}
for grp in [1, 2, 3]:
    sub = df[df['group'] == grp][NET_METRICS].replace([np.inf, -np.inf], np.nan).dropna()
    corr = sub.corr(method='spearman').values
    upper = corr[np.triu_indices_from(corr, k=1)]
    coupling[grp] = np.mean(np.abs(upper))

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Finding 7 — Expertise Tightens the Attentional System', fontsize=13, fontweight='bold')

# Left panel: coupling bar chart
ax = axes[0]
grps = [1, 2, 3]
vals = [coupling[g] for g in grps]
bars = ax.bar([GRP_LBL[g] for g in grps], vals,
              color=[GRP_COL[g] for g in grps], alpha=0.85, edgecolor='white', width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.003, f'{val:.3f}',
            ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Mean |Spearman r| across all metric pairs')
ax.set_title('Coupling Strength by Group\n(higher = more integrated system)', fontsize=10)
ax.set_ylim(0, 0.55)
ax.axhline(0, color='#ccc', linewidth=0.5)

# Right 3 panels: correlation heatmaps per group
for ax, grp in zip(axes[1:], [1, 2, 3]):
    sub = df[df['group'] == grp][NET_METRICS].replace([np.inf, -np.inf], np.nan).dropna()
    corr = sub.corr(method='spearman')
    corr.index = NET_LBL
    corr.columns = NET_LBL
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(corr, ax=ax, cmap='RdBu_r', vmin=-1, vmax=1,
                annot=True, fmt='.2f', annot_kws={'size': 6},
                mask=mask, square=True, linewidths=0.3,
                cbar=(grp == 3))
    ax.set_title(f'{GRP_LBL[grp]} — mean|r|={coupling[grp]:.3f}',
                 fontsize=10, color=GRP_COL[grp], fontweight='bold')
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()